In [ ]:
# Day 25 - LLM as a Judge (Advanced Evaluation)

!pip install -q langchain langchain-community langchain-huggingface

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import torch
from transformers import pipeline

In [ ]:
# Load a Judge Model (can be same or stronger model)
judge_pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=300,
    temperature=0.1   # Low temperature for more consistent judging
)

judge_llm = HuggingFacePipeline(pipeline=judge_pipe)

In [ ]:
# LLM-as-a-Judge Prompt Template
judge_prompt = PromptTemplate(
    input_variables=["question", "context", "answer"],
    template="""You are an expert evaluator. Score the following answer from 1 to 10.

Question: {question}

Context: {context}

Answer: {answer}

Evaluate on these criteria:
1. Faithfulness (Does it stick to the context?) 
2. Relevance (Does it answer the question?)
3. Helpfulness & Clarity

Give a final score (1-10) and short explanation.
Final Score:"""
)

def llm_as_judge(question, context, answer):
    prompt = judge_prompt.format(question=question, context=context, answer=answer)
    result = judge_llm.invoke(prompt)
    print("=== LLM Judge Evaluation ===")
    print(result.strip())
    return result

In [ ]:
# Test the Judge
llm_as_judge(
    question="What is the capital of Ethiopia?",
    context="Addis Ababa is the capital and largest city of Ethiopia.",
    answer="The capital of Ethiopia is Addis Ababa."
)

llm_as_judge(
    question="What is the capital of Ethiopia?",
    context="Addis Ababa is the capital and largest city of Ethiopia.",
    answer="The capital is London."
)